# Gold fact: `fact_project_snapshot`

## Purpose

Publish one row per DPWH project per source snapshot with its reported budget, progress, and every dimension key. This is the main fact for the dashboard and Genie.

## Grain

One DPWH project per distinct source snapshot.

- Uniqueness: `project_key`, `source_snapshot_id`.
- Key: `project_snapshot_key` is `XXHASH64` of the project key and snapshot.
- Rows from older snapshots stay. `is_current_snapshot` marks the newest one.

## What Gold reuses and does not redo

| Value | Comes from |
| --- | --- |
| Budget, progress, dates, status | `02-silver.silver_project` |
| Region and its match status | `02-silver.silver_project_region_map` (Table 13) |
| Flood level, coordinates, and their status | `02-silver.silver_project_flood_map` (Table 15) |
| Snapshot date | `01-bronze.load_log`, the file date of the DPWH snapshot |

Gold runs no region matching, no spatial matching, and no budget consolidation.

## Key `0`

- `region_key = 0` when Table 13 did not match one region. That covers Central Office, unmatched, and ambiguous projects. `region_match_status` keeps the real reason.
- `flood_susceptibility_key = 0` when Table 15 did not publish one level. That never means low or zero risk.
- `status_key = 0` when the source status is missing or conflicting.
- A date key is `0` when the date is missing.

## Delivery rules

- `long_running_flag` follows D-32. It is true when more than 24 months pass from the start date to the completion date. A project without a completion date is measured to the snapshot date instead. It is `NULL` without a start date.
- `zero_progress_flag` comes unchanged from Silver.
- `physical_progress_pct` is published only between 0 and 100. Values outside that range are shifted source values. They become `NULL` here, stay visible in Silver, and the Gold validator counts them.

## Safe reruns

`CREATE TABLE IF NOT EXISTS` and `MERGE` on the deterministic key. Rows are never deleted.

## 1. Confirm upstream safety and select versions

**What:** check that the project, region map, and flood map are each one validated run, built from the same project run, and one row per project.

**Why:** a region or flood result from an older project run would attach to the wrong projects.

**Protects:** every check stops the notebook before any write.

In [ ]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `03-gold`;

DECLARE OR REPLACE VARIABLE project_source_system STRING DEFAULT 'DPWH';
DECLARE OR REPLACE VARIABLE mgb_mapping_source_system STRING DEFAULT 'DENR MGB flood susceptibility';
DECLARE OR REPLACE VARIABLE project_rule_version STRING DEFAULT 'gold-fact-project-snapshot-v1';
-- d-32: long-running means more than 24 months from start to completion, or to the snapshot date
DECLARE OR REPLACE VARIABLE long_running_months INT DEFAULT 24;
DECLARE OR REPLACE VARIABLE delivery_rule_version STRING;

SET VAR delivery_rule_version = CONCAT(
    'gold-delivery-v1|long-running=start-to-completion-or-snapshot>', long_running_months,
    '-months|progress-range=0:100'
);

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(DISTINCT source_snapshot_id) = 1
    AND COUNT(DISTINCT source_ingest_run_id) = 1
    AND COUNT(*) = COUNT(DISTINCT project_key),
    'STOP: silver_project must be nonempty and unique in one selected run'
)
FROM `02-silver`.silver_project;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(dq.action = 'stop' AND dq.status IN ('FAIL', 'ERROR')) = 0,
    'STOP: selected Silver project validation is missing or unsafe'
)
FROM `04-validation`.silver_dq_results AS dq
WHERE dq.table_name IN ('silver_project', 'silver_dpwh_project_component')
  AND dq.pipeline_run_id = (SELECT MAX(project.run_id) FROM `02-silver`.silver_project AS project);

SELECT ASSERT_TRUE(
    COUNT(*) = (SELECT COUNT(*) FROM `02-silver`.silver_project)
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(*) = COUNT(DISTINCT project_key)
    AND MAX(project_run_id) = (SELECT MAX(project.run_id) FROM `02-silver`.silver_project AS project)
    AND COUNT(DISTINCT project_run_id) = 1,
    'STOP: silver_project_region_map must have one row per project from the current project run'
)
FROM `02-silver`.silver_project_region_map;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(dq.action = 'stop' AND dq.status IN ('FAIL', 'ERROR')) = 0,
    'STOP: selected Silver project-region mapping validation is missing or unsafe'
)
FROM `04-validation`.silver_dq_results AS dq
WHERE dq.table_name IN ('silver_project_region_map')
  AND dq.pipeline_run_id = (SELECT MAX(region_map.run_id) FROM `02-silver`.silver_project_region_map AS region_map);

SELECT ASSERT_TRUE(
    COUNT(*) = (SELECT COUNT(*) FROM `02-silver`.silver_project)
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(*) = COUNT(DISTINCT project_key)
    AND MAX(project_run_id) = (SELECT MAX(project.run_id) FROM `02-silver`.silver_project AS project)
    AND COUNT(DISTINCT project_run_id) = 1,
    'STOP: silver_project_flood_map must have one row per project from the current project run'
)
FROM `02-silver`.silver_project_flood_map;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(dq.action = 'stop' AND dq.status IN ('FAIL', 'ERROR')) = 0,
    'STOP: selected Silver project flood-mapping validation is missing or unsafe'
)
FROM `04-validation`.silver_dq_results AS dq
WHERE dq.table_name IN ('silver_project_flood_map')
  AND dq.pipeline_run_id = (SELECT MAX(flood_map.run_id) FROM `02-silver`.silver_project_flood_map AS flood_map);

SELECT ASSERT_TRUE(
    (SELECT COUNT(*) FROM `03-gold`.dim_region WHERE region_key = 0) = 1
    AND (SELECT COUNT(*) FROM `03-gold`.dim_flood_susceptibility WHERE flood_susceptibility_key = 0) = 1,
    'STOP: run the Gold dimension notebooks 00 to 04 before the facts'
);

CREATE OR REPLACE TEMPORARY VIEW project_load_lineage AS
SELECT run_id AS source_ingest_run_id, source_version, source_modified_at,
       COALESCE(completed_at, started_at) AS source_load_ts
FROM (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY run_id
        ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
    ) AS audit_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'dpwh_projects'
      AND status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')
)
WHERE audit_rank = 1;

CREATE OR REPLACE TEMPORARY VIEW selected_project_snapshot AS
SELECT
    MAX(project.run_id) AS project_run_id,
    MAX(project.source_snapshot_id) AS source_snapshot_id,
    MAX(load.source_load_ts) AS project_source_load_ts,
    -- d-32: the snapshot date is the file date of the dpwh snapshot, never the current date
    MAX(DATE(load.source_modified_at)) AS snapshot_date,
    (SELECT MAX(region_map.run_id) FROM `02-silver`.silver_project_region_map AS region_map) AS region_map_run_id,
    (SELECT MAX(flood_map.run_id) FROM `02-silver`.silver_project_flood_map AS flood_map) AS flood_map_run_id
FROM `02-silver`.silver_project AS project
LEFT JOIN project_load_lineage AS load
    ON project.source_ingest_run_id = load.source_ingest_run_id;

SELECT ASSERT_TRUE(
    snapshot_date IS NOT NULL AND project_source_load_ts IS NOT NULL,
    'STOP: the DPWH snapshot has no audited file date or load time in load_log'
)
FROM selected_project_snapshot;

## 2. Create the table

**What:** create `03-gold.fact_project_snapshot` with the column names, types, and order in `docs/gold_model.md`.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS `03-gold`.fact_project_snapshot (
    project_snapshot_key BIGINT NOT NULL COMMENT 'Surrogate key. XXHASH64 of the project key and source snapshot.',
    project_key BIGINT NOT NULL COMMENT 'Project. Joins to dim_project.project_key.',
    region_key BIGINT NOT NULL COMMENT 'Matched official region. Joins to dim_region.region_key. 0 for Central Office, unmatched, and ambiguous projects.',
    status_key INT NOT NULL COMMENT 'Source status. Joins to dim_project_status.status_key. 0 when the status is missing or conflicting.',
    flood_susceptibility_key INT NOT NULL COMMENT 'Flood level at the project point. Joins to dim_flood_susceptibility. 0 means no single approved level, never low or zero risk.',
    snapshot_date_key INT NOT NULL COMMENT 'File date of the DPWH snapshot as yyyymmdd. Joins to dim_date.date_key.',
    start_date_key INT NOT NULL COMMENT 'Project start date as yyyymmdd. Joins to dim_date.date_key. 0 when missing.',
    source_completion_date_key INT NOT NULL COMMENT 'Completion date reported by DPWH as yyyymmdd. Joins to dim_date.date_key. 0 when missing.',
    reported_region_raw STRING COMMENT 'Region text exactly as DPWH reported it. Can be Central Office or an office name.',
    reported_budget_pesos DECIMAL(20,2) COMMENT 'Reported project or contract budget in pesos. Not actual payment or disbursement. NULL when missing or invalid.',
    physical_progress_pct DECIMAL(5,2) COMMENT 'Physical progress from 0 to 100. NULL when missing, conflicting, or outside 0 to 100. Non-additive. Do not sum it.',
    latitude DOUBLE COMMENT 'Project latitude in EPSG:4326. NULL when the source has no agreed coordinate.',
    longitude DOUBLE COMMENT 'Project longitude in EPSG:4326. NULL when the source has no agreed coordinate.',
    coordinate_status STRING COMMENT 'VALID_PAIR, MISSING_PAIR, PARTIAL_PAIR, or OUT_OF_RANGE.',
    region_match_status STRING COMMENT 'Table 13 result: MATCHED, MATCHED_WITH_CONFLICT, NON_GEOGRAPHIC for Central Office, AMBIGUOUS, UNMATCHED, or INVALID_SOURCE.',
    region_match_method STRING COMMENT 'How Table 13 chose the region, such as EXACT_REGION_NAME or COORDINATE_BOUNDARY.',
    region_match_confidence STRING COMMENT 'Table 13 match quality, such as EXACT, SPATIAL, or UNRESOLVED.',
    flood_match_status STRING COMMENT 'Table 15 result: MATCHED, AMBIGUOUS, UNMATCHED, or INVALID_SOURCE.',
    long_running_flag BOOLEAN COMMENT 'True when more than 24 months pass from start to completion, or to the snapshot date when not completed. NULL without a start date.',
    zero_progress_flag BOOLEAN COMMENT 'True when resolved physical progress is exactly zero. NULL when progress is unresolved.',
    delivery_rule_version STRING COMMENT 'Version of the long-running and progress rules.',
    run_id STRING COMMENT 'Deterministic Gold run ID. Hash of the Silver project, region map, and flood map runs plus the rule versions.',
    is_current_snapshot BOOLEAN COMMENT 'True for the newest DPWH snapshot. Filter on it to avoid counting a project twice.',
    source_load_ts TIMESTAMP COMMENT 'Latest load time of the project, region, and flood inputs behind this row.',
    source_snapshot_id STRING COMMENT 'DPWH source snapshot of this row.'
)
USING DELTA
COMMENT 'Gold project fact. One DPWH project per source snapshot with reported budget, progress, and keys for project, region, status, flood level, and dates. Filter is_current_snapshot for current totals.';

## 3. Resolve every key

**What:** join each Silver project to its Table 13 and Table 15 results and to the Gold dimensions. Turn each date into a `yyyymmdd` key.

**Why:**

- A region key is used only for `MATCHED` and `MATCHED_WITH_CONFLICT`. Every other region result goes to key `0`.
- A flood key is used only for `MATCHED`. Every other flood result goes to key `0`.
- The status key comes from the active row of the current mapping version.

**Protects:** the notebook stops if a project has no `dim_project` row, if a date key is missing from `dim_date`, or if keys repeat.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW fact_project_snapshot_source AS
WITH project_rows AS (
    SELECT
        project.contract_id,
        project.status_raw,
        project.start_date,
        project.completion_date,
        project.reported_budget_pesos,
        project.physical_progress_pct,
        project.zero_progress_flag,
        project.source_snapshot_id,
        region_map.reported_region_raw,
        region_map.psgc_region_code,
        region_map.psgc_version,
        region_map.match_status AS region_match_status,
        region_map.mapping_method AS region_match_method,
        region_map.match_quality AS region_match_confidence,
        region_map.source_load_ts AS region_source_load_ts,
        flood_map.latitude,
        flood_map.longitude,
        flood_map.coordinate_status,
        flood_map.flood_susceptibility_level,
        flood_map.mgb_mapping_source_version,
        flood_map.match_status AS flood_match_status,
        flood_map.source_load_ts AS flood_source_load_ts
    FROM `02-silver`.silver_project AS project
    INNER JOIN `02-silver`.silver_project_region_map AS region_map
        ON project.project_key = region_map.project_key
    INNER JOIN `02-silver`.silver_project_flood_map AS flood_map
        ON project.project_key = flood_map.project_key
),
keyed AS (
    SELECT
        dim_project.project_key,
        CASE
            WHEN project_rows.region_match_status IN ('MATCHED', 'MATCHED_WITH_CONFLICT')
                THEN region.region_key
            ELSE 0
        END AS region_key,
        COALESCE(status.status_key, 0) AS status_key,
        CASE
            WHEN project_rows.flood_match_status = 'MATCHED' THEN flood.flood_susceptibility_key
            ELSE 0
        END AS flood_susceptibility_key,
        CAST(DATE_FORMAT(selected.snapshot_date, 'yyyyMMdd') AS INT) AS snapshot_date_key,
        COALESCE(CAST(DATE_FORMAT(project_rows.start_date, 'yyyyMMdd') AS INT), 0) AS start_date_key,
        COALESCE(CAST(DATE_FORMAT(project_rows.completion_date, 'yyyyMMdd') AS INT), 0)
            AS source_completion_date_key,
        project_rows.reported_region_raw,
        CAST(project_rows.reported_budget_pesos AS DECIMAL(20, 2)) AS reported_budget_pesos,
        CASE
            WHEN project_rows.physical_progress_pct BETWEEN 0 AND 100
                THEN CAST(project_rows.physical_progress_pct AS DECIMAL(5, 2))
        END AS physical_progress_pct,
        project_rows.latitude,
        project_rows.longitude,
        project_rows.coordinate_status,
        project_rows.region_match_status,
        project_rows.region_match_method,
        project_rows.region_match_confidence,
        project_rows.flood_match_status,
        CASE
            WHEN project_rows.start_date IS NULL THEN NULL
            ELSE ADD_MONTHS(project_rows.start_date, long_running_months) < LEAST(
                COALESCE(project_rows.completion_date, selected.snapshot_date), selected.snapshot_date
            )
        END AS long_running_flag,
        project_rows.zero_progress_flag,
        delivery_rule_version,
        SHA2(CONCAT_WS(
            '|', selected.project_run_id, selected.region_map_run_id, selected.flood_map_run_id,
            delivery_rule_version, project_rule_version
        ), 256) AS run_id,
        TRUE AS is_current_snapshot,
        GREATEST(
            selected.project_source_load_ts,
            project_rows.region_source_load_ts,
            project_rows.flood_source_load_ts
        ) AS source_load_ts,
        project_rows.source_snapshot_id
    FROM project_rows
    CROSS JOIN selected_project_snapshot AS selected
    LEFT JOIN `03-gold`.dim_project AS dim_project
        ON dim_project.source_system = project_source_system
       AND dim_project.contract_id = project_rows.contract_id
    LEFT JOIN `03-gold`.dim_region AS region
        ON project_rows.psgc_region_code = region.psgc_region_code
       AND project_rows.psgc_version <=> region.psgc_version
    LEFT JOIN `03-gold`.dim_project_status AS status
        ON status.source_system = project_source_system
       AND status.source_status = project_rows.status_raw
       AND status.is_active
       AND status.status_key <> 0
    LEFT JOIN `03-gold`.dim_flood_susceptibility AS flood
        ON flood.source_system = mgb_mapping_source_system
       AND flood.flood_susceptibility_level = project_rows.flood_susceptibility_level
       AND flood.source_version = project_rows.mgb_mapping_source_version
)
SELECT
    XXHASH64(CONCAT_WS(
        '|', 'PROJECT_SNAPSHOT', CAST(project_key AS STRING), source_snapshot_id
    )) AS project_snapshot_key,
    *
FROM keyed;

SELECT ASSERT_TRUE(
    COUNT(*) = (SELECT COUNT(*) FROM `02-silver`.silver_project)
    AND COUNT_IF(project_key IS NULL) = 0
    AND COUNT_IF(region_key IS NULL) = 0
    AND COUNT_IF(flood_susceptibility_key IS NULL) = 0
    AND COUNT(*) = COUNT(DISTINCT project_snapshot_key)
    AND COUNT(*) = COUNT(DISTINCT NAMED_STRUCT('project', project_key, 'snapshot', source_snapshot_id)),
    'STOP: every project needs one fact row with a project, region, and flood key'
)
FROM fact_project_snapshot_source;

SELECT ASSERT_TRUE(
    COUNT(*) = 0,
    'STOP: a fact date key has no row in dim_date. Widen the dim_date range.'
)
FROM (
    SELECT fact.snapshot_date_key AS date_key FROM fact_project_snapshot_source AS fact
    UNION ALL
    SELECT fact.start_date_key FROM fact_project_snapshot_source AS fact
    UNION ALL
    SELECT fact.source_completion_date_key FROM fact_project_snapshot_source AS fact
) AS used_dates
LEFT ANTI JOIN `03-gold`.dim_date AS calendar
    ON used_dates.date_key = calendar.date_key;

## 4. Merge and mark the current snapshot

**What:** merge the rows by key. Then set `is_current_snapshot` to true only for the newest DPWH snapshot.

**Why:** older snapshot rows stay for history. Only one snapshot per project is current, so current totals never count a project twice.

**Expected validation:** at most one current row per project.

In [ ]:
%sql
MERGE INTO `03-gold`.fact_project_snapshot AS target
USING fact_project_snapshot_source AS source
ON target.project_snapshot_key = source.project_snapshot_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

UPDATE `03-gold`.fact_project_snapshot
SET is_current_snapshot = (
    source_snapshot_id = (SELECT selected.source_snapshot_id FROM selected_project_snapshot AS selected)
)
WHERE is_current_snapshot <> (
    source_snapshot_id = (SELECT selected.source_snapshot_id FROM selected_project_snapshot AS selected)
);

SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT project_key),
    'STOP: a project has more than one current snapshot row'
)
FROM `03-gold`.fact_project_snapshot
WHERE is_current_snapshot;

# Summary

Created or refreshed `03-gold.fact_project_snapshot` with one row per DPWH project for the current snapshot. Budgets reconcile to `silver_project`. Region, flood, and status results that are not a single clean match use key `0`, and their real reason stays in the match status columns. The query below shows the coverage. Next: `04_validation/09_validation_gold`.

In [ ]:
%sql
SELECT
    region_match_status,
    flood_match_status,
    COUNT(*) AS projects,
    SUM(reported_budget_pesos) AS reported_budget_pesos,
    COUNT_IF(region_key = 0) AS projects_on_region_key_0,
    COUNT_IF(long_running_flag) AS long_running_projects
FROM `03-gold`.fact_project_snapshot
WHERE is_current_snapshot
GROUP BY ALL
ORDER BY projects DESC;